# Import

In [ ]:
import numpy as np
from scipy.stats import norm
import utils

import matplotlib.pyplot as plt
import plotly.graph_objs as go
import matplotlib.pylab as pylab
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

import re

import pickle

from importlib import reload

import tqdm

In [2]:
params_plot = {'legend.fontsize': 'x-large',
          'figure.figsize': (8, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params_plot)
%config InlineBackend.figure_format = "retina"

plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{amsmath,amsfonts}')
dpi = 300

fs = 14
fsL = 19
plt.rcParams["font.size"] = fs

# AMM Simulation Process

In [3]:
reload(utils)

<module 'utils' from '/Users/sebastien/Desktop/AMM-Optimal-Exit-Time/code/v2/utils.py'>

## Main functions to run and plot models

In [4]:
def run(params_glob, params_grid, params_LP, time_factor=0, jump_slice:np.array=np.array([1000]), LS=True, PDE=True, risk_neutral_bool=True):

    res_euler_dict, value_function_LS_dict = {}, {}

    if risk_neutral_bool:
        risk_type = 'Risk Neutral' 
    else:
        risk_type = 'Risk Averse'

    if PDE:
        amm_solver_obj = utils.SolverV2(**{**params_glob, **params_grid, **params_LP})

        # Solving the AMM problem using Euler method
        print(f'Solving the AMM problem using Euler method - {risk_type}', end='\n\n')
        res_euler_dict = amm_solver_obj.euler(delta=params_grid['delta'], h=params_grid['h'], jump_scale_nbr=params_grid['jump_scale_nbr'], S_scale_factor=params_grid['S_scale_factor'], risk_neutral=risk_neutral_bool)
        print('--> Done', end='\n\n')

        del amm_solver_obj

    if LS:
        # Solving the AMM problem using Longstaff-Schwartz method
        print(f'Solving the AMM problem using Longstaff-Schwartz method - {risk_type}', end='\n\n')

        # Create the solver object
        params = {**params_glob, **params_grid, **params_LP}
        initial_amm_solver_obj = utils.SolverV2(**params)

        # Create a range of S0 values based on the scale factor
        S0_matrix = np.arange(params['S0']*(1-params['S_scale_factor']), params['S0']*(1+params['S_scale_factor']) + params['h'], params['h'])
        Y0_matrix = jump_slice

        value_function_LS_dict = {'Y0' : Y0_matrix,
                                'value_function_LS' : np.zeros((Y0_matrix.shape[0], S0_matrix.shape[0])),
                                'confidence_intervals' : np.zeros((Y0_matrix.shape[0], S0_matrix.shape[0], 2))}
        
        # Generate paths
        initial_amm_solver_obj.amm_model(BM_type=params_glob['BM_type'])
        full_amm_paths = initial_amm_solver_obj.get_paths()
        initial_amm_paths_S = full_amm_paths['external_mid_price_S']
        initial_amm_paths_Y = full_amm_paths['asset_Y']
        
        j = 0
        for Y0_val in Y0_matrix:
            
            value_function_LS = np.zeros_like(S0_matrix)
            params['new_Y0'] = Y0_val
            print(f'LS for Y0 = {params['new_Y0']}', end='\n')

            # Loop over different values of S0
            for i in tqdm.tqdm(range(S0_matrix.shape[0])):

                params['S0'] = S0_matrix[i]

                # Create the solver object
                amm_solver_obj = utils.SolverV2(**params)

                # Generate paths
                amm_solver_obj.amm_model(BM_type=params_glob['BM_type'])
                full_amm_paths = amm_solver_obj.get_paths()
                amm_paths = full_amm_paths['amm_model_0']
                #amm_paths_S = full_amm_paths['external_mid_price_S']
                #amm_paths_Y = full_amm_paths['asset_Y']

                # Calculate the value function at time
                res_LS = amm_solver_obj.longstaff_schwartz(paths=amm_paths, 
                                                           paths_S=initial_amm_paths_S, 
                                                           paths_Y=initial_amm_paths_Y, 
                                                           deg=params_glob['deg'], risk_neutral=risk_neutral_bool)

                value_function_LS[i] = res_LS['V_matrix'][:, -1 if time_factor == 1 else int(params['n_steps']*time_factor)+1].mean()

                # Compute confidence interval 5%
                sigma_mc = np.std(res_LS['V_matrix'][:, 1], ddof=1)
                value_function_LS_dict['confidence_intervals'][j, i, 0] = value_function_LS[i] - 1.96 * sigma_mc/np.sqrt(params['n_steps'])
                value_function_LS_dict['confidence_intervals'][j, i, 1] = value_function_LS[i] + 1.96 * sigma_mc/np.sqrt(params['n_steps'])

                del amm_solver_obj

            value_function_LS_dict['value_function_LS'][j, :] = value_function_LS
            j += 1

        print('--> Done', end='\n\n')

    # Save the results in a dictionary
    all_dicts = {'value_function_LS_dict': value_function_LS_dict, 'res_euler_dict': res_euler_dict}

    return all_dicts

In [5]:
def plot(data_LS, data_euler, params, risk_type):

    fig = go.Figure()

    risk_type_color = {'risk_neutral_euler':'blue',
                       'risk_neutral_LS':'green',
                       'risk_averse_euler':'orange',
                       'risk_averse_LS':'purple'}

    if data_euler:
        # Plot models
        # Plot Euler
        space, jumps = np.meshgrid(data_euler['external_mid_price_S'], data_euler['jumps_grid'])

        x = jumps.flatten()
        y = space.flatten()
        z = data_euler['V_matrix'][0, :, :].flatten()

        fig.add_trace(go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='markers',
            marker=dict(size=2, color=risk_type_color[f'{risk_type}_euler'], opacity=0.8),
            name=f'Euler - {risk_type}'
        ))

        x=data_euler['jumps_grid'].flatten()

        # Plot local price
        fig.add_trace(go.Scatter3d(
            x=x,
            y=params['Y0']*params['X0']/x**2,
            z=np.zeros_like(data_euler['V_matrix'][0, :, :].flatten()),
            mode='lines',
            line=dict(color='red', width=2),
            name='Local Price',
        ))


    if data_LS:
        # Plot LS for different slice
        for k in range(data_LS['Y0'].shape[0]):

            S0_matrix = np.arange((1-params['S_scale_factor'])*params['S0'], (1+params['S_scale_factor'])*params['S0'] + params['h'], params['h'])

            fig.add_trace(go.Scatter3d(
                x=np.ones_like(S0_matrix)*data_LS['Y0'][k],
                y=S0_matrix,
                z=data_LS['value_function_LS'][k, :],
                mode='lines+markers',
                marker=dict(color=risk_type_color[f'{risk_type}_LS'], size=2),
                name=f'LS Line Slice - {risk_type} - {data_LS['Y0'][k]}'
            ))

    return fig

## Run model

In [6]:
# Parameters

S0 = 1_000; Y0 = 1_000; trade_size = 1; fee = 0.1 * trade_size * S0

params_glob = {'sigma': 100, 'rate': 0, 'T': 1, 'S0': S0, 'deg':2, 'BM_type': 'arithmetic'} 
params_LP = {'a0': 4. , 'a1': 8., 'a2': 0.04, 'ksi': trade_size, 'X0': Y0 * S0, 'Y0': Y0, 'new_Y0': None, 'psi': 10**(-3), 'fees_coeff': fee}
params_LP['c'] = params_LP['X0'] * params_LP['Y0']  # Ensure c is set correctly
params_grid = {'n_paths': 5_000, 'n_steps': 1_440, 'S_scale_factor': 0.3, 'jump_scale_nbr': 50, 'h':10, 'delta': 0.001}

runned = False
res_glob = {}

############## Risk-neutral ###############
LS, PDE = True, True
res_glob['risk_neutral_dict'] = run(params_glob, params_grid, params_LP, jump_slice=np.array([1000]), LS=LS, PDE=PDE, risk_neutral_bool=True)
# Save
if LS:
    with open(f'saved_pkl/LS_model_risk_neutral.pkl', 'wb') as f:
        pickle.dump(res_glob['risk_neutral_dict']['value_function_LS_dict'], f)
    f.close()
    runned = True
if PDE:
    with open(f'saved_pkl/euler_model_risk_neutral.pkl', 'wb') as f:
        pickle.dump(res_glob['risk_neutral_dict']['res_euler_dict'], f)
    f.close()
    runned = True

############### Risk-averse ###############
LS, PDE = True, True
res_glob['risk_averse_dict'] = run(params_glob, params_grid, params_LP, jump_slice=np.array([1000]), LS=LS, PDE=PDE, risk_neutral_bool=False)
# Save
if LS:
    with open(f'saved_pkl/LS_model_risk_averse.pkl', 'wb') as f:
        pickle.dump(res_glob['risk_averse_dict']['value_function_LS_dict'], f)
    f.close()
    runned = True
if PDE:
    with open(f'saved_pkl/euler_model_risk_averse.pkl', 'wb') as f:
        pickle.dump(res_glob['risk_averse_dict']['res_euler_dict'], f)
    f.close()

if runned:
    with open(f'saved_pkl/params.pkl', 'wb') as f:
        pickle.dump({**params_glob, **params_grid, **params_LP}, f)
    f.close()

Solving the AMM problem using Euler method - Risk Neutral



100%|██████████| 1000/1000 [00:08<00:00, 116.04it/s]


--> Done

Solving the AMM problem using Longstaff-Schwartz method - Risk Neutral

LS for Y0 = 1000


100%|██████████| 61/61 [04:07<00:00,  4.05s/it]


--> Done

Solving the AMM problem using Euler method - Risk Averse



100%|██████████| 1000/1000 [03:51<00:00,  4.32it/s]


--> Done

Solving the AMM problem using Longstaff-Schwartz method - Risk Averse

LS for Y0 = 1000


100%|██████████| 61/61 [05:36<00:00,  5.51s/it]

--> Done



## Load the results

In [7]:
res_dict_risk_neutral, res_dict_risk_averse = {}, {}

############### Risk-neutral ###############
try:
    with open(f'saved_pkl/LS_model_risk_neutral.pkl', 'rb') as f:
        res_dict_risk_neutral['value_function_LS_dict'] = pickle.load(f)
        print(f'Risk neutral dict with LS ---> loaded')
except Exception as e:
    print(f'! Error : {e}')

try:
    with open(f'saved_pkl/euler_model_risk_neutral.pkl', 'rb') as f:
        res_dict_risk_neutral['res_euler_dict'] = pickle.load(f)
        print(f'Risk neutral dict with Euler ---> loaded')
except Exception as e:
    print(f'! Error : {e}')

############### Risk-averse ###############
try:
    with open(f'saved_pkl/LS_model_risk_averse.pkl', 'rb') as f:
        res_dict_risk_averse['value_function_LS_dict'] = pickle.load(f)
        print(f'Risk averse dict with LS ---> loaded')
except Exception as e:
    print(f'! Error : {e}')

try: 
    with open(f'saved_pkl/euler_model_risk_averse.pkl', 'rb') as f:
        res_dict_risk_averse['res_euler_dict'] = pickle.load(f)
        print(f'Risk averse dict with Euler ---> loaded')
except Exception as e:
    print(f'! Error : {e}')

############### Params ###############
try: 
    with open(f'saved_pkl/params.pkl', 'rb') as f:
        params = loaded_dicts = pickle.load(f)
    f.close()
    print(f'Parameters ---> loaded')

except Exception as e:
    print(f'! Error : {e}')

Risk neutral dict with LS ---> loaded
Risk neutral dict with Euler ---> loaded
Risk averse dict with LS ---> loaded
Risk averse dict with Euler ---> loaded
Parameters ---> loaded


## Plot the results (analyse)

In [8]:
fig_risk_neutral = plot(res_dict_risk_neutral['value_function_LS_dict'], res_dict_risk_neutral['res_euler_dict'], params, risk_type='risk_neutral')
fig_risk_averse = plot(res_dict_risk_averse['value_function_LS_dict'], res_dict_risk_averse['res_euler_dict'], params, risk_type='risk_averse')

fig = go.Figure()
fig.add_traces(fig_risk_neutral.data + fig_risk_averse.data)

fig.update_layout(
    width=1000,
    height=900,
    scene=dict(xaxis_title='Y', yaxis_title='S', zaxis_title='Value',
    ),
    scene_camera=dict(
    eye=dict(x=1.6, y=1.5, z=1.5),
    center=dict(x=0, y=0, z=-0.2) 
    ),
    title = f'Value Function of AMM Model by Euler Method',
)

fig.write_html(f'figures/3d-euler-vs-ls.html')
fig.show()

## Plots the results (paper)

## 3D Plot of Euler Surface at different time t

In [ ]:
time_factor_list = [0, 0.5, 0.9]
color_list = ['blue', 'green', 'purple']
alpha_list = [1, 1, 1]

res_dict_risk_neutral['res_euler_dict']

space, jumps = np.meshgrid(res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'], res_dict_risk_neutral['res_euler_dict']['jumps_grid'])
value_function_euler = res_dict_risk_neutral['res_euler_dict']['V_matrix']

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Prepare data for plotting
x = jumps.flatten()
y = space.flatten()

for time_factor_k, color_k, alpha_k in zip(time_factor_list, color_list, alpha_list):

    z = value_function_euler[int(value_function_euler.shape[0]*time_factor_k), :, :]
    scatter = ax.scatter(x, y, z.flatten(), color=color_k, alpha=alpha_k, marker='x', s=0.1, label='Euler')
    #surface = ax.plot_surface(jumps, space, z, color=color_k, alpha=alpha_k, edgecolor='none')

# Local Price Line
x_lp = np.linspace(min(x), max(x), len(x))
y_lp = params['Y0'] * params['X0'] / x_lp**2
z_lp = np.zeros_like(x_lp)
ax.plot(x_lp, y_lp, z_lp, color='black', linewidth=2, label='Local Price')

# Title, labels, and adjustments
ax.set_xlabel(r'$Y$', fontsize=fsL,labelpad=8)
ax.set_ylabel(r'$S$', fontsize=fsL,labelpad=8)
ax.set_zlabel(r'$v$', fontsize=fsL,labelpad=8)

ax.tick_params(axis='x', labelsize=fs)
ax.tick_params(axis='y', labelsize=fs)
ax.tick_params(axis='z', labelsize=fs)

ax.xaxis.set_major_locator(MaxNLocator(nbins=4))

ax.view_init(elev=12, azim=-20)

fig.subplots_adjust(top=0.99, bottom=0.01)

# Show the plot
plt.savefig(f'figures/3d-plot-euler_through_time.pdf', dpi=dpi)
plt.show()


## 2D Plot of Euler Surface at different time t

In [ ]:
time_factor_list = [0, 0.5, 0.9]
color_list = ["#2E55D5", "#309F55", "#9D5AD4"]
alpha_list = [0.9, 0.9, 0.9]

space = res_dict_risk_neutral['res_euler_dict']['external_mid_price_S']
jumps = res_dict_risk_neutral['res_euler_dict']['jumps_grid']
value_function_euler = res_dict_risk_neutral['res_euler_dict']['V_matrix']

x = space
y = jumps
exercise_color = "#F2EDED"

mosaic = [["T0", "T1", "T2", "B"]]

fig, axd = plt.subplot_mosaic(mosaic, figsize=(17, 3), 
                              gridspec_kw={'wspace': 0.5})

for i, (time_factor_k, color_k, alpha_k) in enumerate(zip(time_factor_list, color_list, alpha_list)):

    idx = int(value_function_euler.shape[0] * time_factor_k)
    if idx >= value_function_euler.shape[0]: 
        idx = value_function_euler.shape[0] - 1
    
    z = value_function_euler[idx, :, :]
    z_bool = np.where(z > 0, 0, np.nan)
    cmap_binary = ListedColormap([color_k, exercise_color])
    
    # Main plot
    axd['B'].pcolormesh(x, y, z_bool, cmap=cmap_binary, alpha=alpha_k, shading='auto')
    
    # Individual plots
    ax_b = axd[f'T{i}']
    im_b = ax_b.pcolormesh(x, y, z_bool, cmap=cmap_binary, alpha=alpha_k, shading='auto')

    # Style des petits plots
    ax_b.set_facecolor(exercise_color)
    ax_b.set_title(r'$t=$'+f' {time_factor_k}', fontsize=fs+2)
    
    # Color scale
    im_b.set_clim(0, 1) 
    cbar = fig.colorbar(im_b, ax=ax_b, ticks=[0.25, 0.75], fraction=0.046, pad=0.02)
    cbar.ax.set_yticklabels([r'continuation', r'exit'], rotation=90, va='center', fontsize=fs)
    cbar.ax.tick_params(length=0)

axd['B'].set_facecolor(exercise_color)
axd['B'].set_title(r'$\mathrm{combined~}\mathrm{overlays}$', fontsize=fs)

# Local price c/y**2
y_lp = np.concatenate((np.array([np.min(y)-0.5]), y, np.array([np.max(y)+0.5] ))) 
x_lp = params['Y0'] * params['X0'] / y_lp**2

for k in axd:
    axd[k].plot(x_lp, y_lp, color='black', linewidth=1.5, linestyle='--', label='Local Price' if k == 'B' else "")
    axd[k].set_xlabel(r'$S$', fontsize=fsL)
    axd[k].set_ylabel(r'$Y$', fontsize=fsL)
    axd[k].xaxis.set_major_locator(MaxNLocator(nbins=5))
    axd[k].yaxis.set_major_locator(MaxNLocator(nbins=5))
    axd[k].tick_params(axis='both', labelsize=fs)

# Save and plot
plt.tight_layout()
plt.savefig(f'figures/2d-plot-euler_through_time.pdf', dpi=dpi, bbox_inches='tight')
plt.show()

## 3D Plot to compare Euler and LS methods at time t=0 (multiple slices)

In [ ]:
space, jumps = np.meshgrid(res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'], res_dict_risk_neutral['res_euler_dict']['jumps_grid'])

# Euler
value_function_euler = res_dict_risk_neutral['res_euler_dict']['V_matrix']

# LS
value_function_LS_dict = res_dict_risk_neutral['value_function_LS_dict']

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Prepare data for plotting
x = jumps.flatten()
y = space.flatten()

for time_factor_k, color_k, alpha_k in zip([0, 0.5, 0.9], ['blue', 'green', 'purple'], [1, 1, 1]):

    z = value_function_euler[int(value_function_euler.shape[0]*time_factor_k), :, :]
    scatter = ax.scatter(x, y, z.flatten(), color=color_k, alpha=alpha_k, marker='x', s=0.1, label='Euler')
    #surface = ax.plot_surface(jumps, space, z, color=color_k, alpha=alpha_k, edgecolor='none')

# Local Price Line
x_lp = np.linspace(min(x), max(x), len(x))
y_lp = params['Y0'] * params['X0'] / x_lp**2
z_lp = np.zeros_like(x_lp)
ax.plot(x_lp, y_lp, z_lp, color='black', linewidth=2, label='Local Price')

# LS Line Slices
for k in range(value_function_LS_dict['Y0'].shape[0]):
    ax.plot(np.ones_like(res_dict_risk_neutral['res_euler_dict']['external_mid_price_S']) * value_function_LS_dict['Y0'][k], 
            res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'], 
            value_function_LS_dict['value_function_LS'][k, :], 
            color='red', marker='.',linewidth=1, label='LS Line Slice')
    
# Add rectangular slices at certain jump levels
slice_jumps = value_function_LS_dict['Y0']
y_min, y_max = np.min(res_dict_risk_neutral['res_euler_dict']['external_mid_price_S']), np.max(res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'])
z_min, z_max = 0, np.max(res_dict_risk_neutral['res_euler_dict']['V_matrix']) + 100

for j in slice_jumps:
    verts = [[
        (j, y_min, z_min),
        (j, y_max, z_min),
        (j, y_max, z_max),
        (j, y_min, z_max)
    ]]
    rect = Poly3DCollection(verts, facecolors='black', alpha=0.03, edgecolors='gray', linewidths=1)
    ax.add_collection3d(rect)

# Title and labels
ax.set_xlabel(r'$Y$', fontsize=fsL,labelpad=8)
ax.set_ylabel(r'$S$', fontsize=fsL,labelpad=8)
ax.set_zlabel(r'$v$', fontsize=fsL,labelpad=8)

# Adjust tick sizes for X, Y, and Z axes
ax.tick_params(axis='x', labelsize=fs)
ax.tick_params(axis='y', labelsize=fs)
ax.tick_params(axis='z', labelsize=fs)

ax.xaxis.set_major_locator(MaxNLocator(nbins=4))

#ax.view_init(elev=12, azim=-20)

fig.subplots_adjust(top=0.99, bottom=0.01)

# Save the plot
plt.savefig(f'figures/3d-plot_euler_vs_LS.pdf', dpi=dpi)
plt.show()


## 3D Plot to compare Euler surface in risk-neutral and risk-averse case

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

space, jumps = np.meshgrid(res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'], res_dict_risk_neutral['res_euler_dict']['jumps_grid'])

x = jumps.flatten()
y = space.flatten()

# Euler risk-neutral
z =  res_dict_risk_neutral['res_euler_dict']['V_matrix'][0, :, :]
scatter = ax.scatter(x, y, z.flatten(), color='blue', alpha=0.5, marker='x', s=0.1, label='Euler - Risk-Neutral')
#surface = ax.plot_surface(jumps, space, z, color=color_k, alpha=alpha_k, edgecolor='none')

# Euler risk-averse
z = res_dict_risk_averse['res_euler_dict']['V_matrix'][0, :, :]
scatter = ax.scatter(x, y, z.flatten(), color='#e66101', alpha=0.9, marker='x', s=0.1, label='Euler - Risk-Averse')
#surface = ax.plot_surface(jumps, space, z, color=color_k, alpha=alpha_k, edgecolor='none')

# Local Price Line
x_lp = np.linspace(min(x), max(x), len(x))
y_lp = params['Y0'] * params['X0'] / x_lp**2
z_lp = np.zeros_like(x_lp)
ax.plot(x_lp, y_lp, z_lp, color='black', linewidth=1.5, label='Local Price')

# Title and labels
ax.set_xlabel(r'$Y$', fontsize=fsL,labelpad=8)
ax.set_ylabel(r'$S$', fontsize=fsL,labelpad=8)
ax.set_zlabel(r'$v$', fontsize=fsL,labelpad=8)

# Adjust tick sizes for X, Y, and Z axes
ax.tick_params(axis='x', labelsize=fs)
ax.tick_params(axis='y', labelsize=fs)
ax.tick_params(axis='z', labelsize=fs)

ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

fig.subplots_adjust(top=0.99, bottom=0.01)

# Save the plot
ax.view_init(elev=14, azim=-213, roll=0.5, vertical_axis='z')
plt.savefig(f'figures/3d-plot-euler-risk_averse_vs_neutral.pdf', dpi=dpi)


# 2D Plots

## 2D Plot to compare Euler and LS methods at time t=0 (with confidence interval) (multiple slices)

In [ ]:
for i in range(value_function_LS_dict['Y0'].shape[0]):
    
    fig, axs = plt.subplots(1, 1, sharex=True, figsize=(8,6))
    #fig.tight_layout()
    
    jump_index = int(np.where(res_dict_risk_neutral['res_euler_dict']['jumps_grid'] == value_function_LS_dict['Y0'][i])[0][0])

    plt.scatter(x=res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'],y=value_function_euler[0, jump_index, :], label=r'$v_{\mathrm{Euler}}$', color ='tab:blue', marker='x')
    plt.scatter(x=res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'],y=value_function_LS_dict['value_function_LS'][i, :], label=r'$v_{\mathrm{LS}}$', color='tab:red', marker='.')
    plt.fill_between(x=res_dict_risk_neutral['res_euler_dict']['external_mid_price_S'],y1=value_function_LS_dict['confidence_intervals'][i, :, 0], y2=value_function_LS_dict['confidence_intervals'][i, :, 1], color='tab:red', alpha=0.25)
    plt.xlabel(r'$S$~~$\mathrm{(USDC/ETH)}$', fontsize = fsL+3)
    plt.ylabel(r'$v$', fontsize = fsL+3)
    plt.annotate(r'$Y_0 = ' + str(value_function_LS_dict['Y0'][i]) + '$', xy=(1180, y_max-50), xytext=(1, 0), textcoords='offset points', fontsize=fsL+3)
    plt.legend(fontsize=fsL+4, loc='upper left')

    plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True, prune='lower', nbins=8))
    plt.ylim(-50,y_max + 170)

    fig.subplots_adjust(bottom=0.15)
    plt.savefig(f'figures/2d-plot_euler_vs_LS-{value_function_LS_dict['Y0'][i]}.pdf', dpi=dpi)

    plt.show()